# Снижение размерности: SVD → PCA → t-SNE

> Модуль **Unsupervised Learning** (обучение без учителя). Первый ноутбук — про то, как
> взять данные в десятках и сотнях измерений и **увидеть** их структуру в 2D/3D, не потеряв
> сути.

### План занятия
1. **Зачем** снижать размерность — проклятие размерности, визуализация, шумоподавление, скорость.
2. **SVD** — сингулярное разложение, общий «движок» под капотом. Геометрия и низкоранговое приближение.
3. **PCA** — метод главных компонент как SVD на центрированных данных. Объяснённая дисперсия, scree-plot, реконструкция.
4. **t-SNE** — нелинейное вложение, сохраняющее соседство. Perplexity и подводные камни.
5. **Реальный пример** — 128-мерные WavLM-эмбеддинги трёх дикторов: увидим их в 2D и 3D.

Дальше, в следующих ноутбуках, мы будем эти структуры **искать алгоритмически** —
это и есть кластеризация.

## 1. Зачем снижать размерность — практический взгляд

Каждый объект в данных описывается набором чисел — **признаками**. Их бывает очень много:

- один короткий аудиофрагмент → **128 чисел** (эмбеддинг голоса, как в нашем датасете);
- маленькая картинка 28×28 → **784 числа** (по одному на пиксель);
- профиль пользователя или товар → сотни характеристик.

Каждое число — это отдельное **измерение** (dimension). Человек легко представляет 2–3
измерения, но «облако точек в 128-мерном пространстве» вообразить невозможно. **Снижение
размерности** — это способ описать те же объекты *меньшим* числом чисел, сохранив суть.
Грубо говоря — сделать короткую, но ёмкую выжимку.

### Когда об этом стоит задуматься (практические ситуации)

- **Хочется увидеть данные глазами.** 128 измерений не нарисовать; сжать до 2D/3D — и сразу
  видно группы, выбросы, структуру. (Именно это мы сделаем с голосами дикторов.)
- **Модель учится медленно или ест слишком много памяти.** Меньше признаков → быстрее
  обучение и меньше места. Особенно когда признаков тысячи (тексты, картинки).
- **Много дублирующих признаков.** «Рост в см» и «рост в дюймах» несут одно и то же. Снижение
  размерности схлопывает такие дубли в одно направление.
- **В данных много шума.** Полезный сигнал часто живёт в небольшом числе направлений, а
  остальное — случайные колебания. Отбросив «мелкие» направления, мы чистим данные.
- **Поиск похожих, рекомендации, дедупликация.** Компактные векторы → быстрый поиск ближайших
  соседей (похожие лица, голоса, товары).
- **Сжатие** изображений и сигналов.

### Что это даёт и чем рискуем

| ✅ Что даёт | ⚠️ Риски и ограничения |
|---|---|
| наглядность — видно структуру | **теряется часть информации** — вопрос лишь, какая именно |
| скорость и экономию памяти | **теряется интерпретируемость**: новые оси — это *смеси* старых признаков, их трудно назвать словом |
| шумоподавление, иногда лучше качество | **линейные методы не видят «кривую» структуру** (увидим на «рулете») |
| компактные векторы для поиска похожих | **утечка (leakage)**: преобразование «обучают» только на train, иначе подсматриваем тест |
| | **t-SNE / UMAP — только для картинки**: по ним нельзя мерить расстояния и кластеризовать |

> **Главная мысль.** Снижение размерности — это всегда **компромисс**: мы меняем часть
> информации на компактность и наглядность. Дальше разберём, *как* именно выбирают, что
> сохранить, а что выбросить, — начиная с самого универсального инструмента, **SVD**.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from sklearn.datasets import load_digits, make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

try:
    from ipywidgets import interact, IntSlider, Dropdown
    HAVE_WIDGETS = True
except Exception:
    HAVE_WIDGETS = False

np.random.seed(0)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True, "grid.alpha": 0.3})
PALETTE = ["#2f6f4f", "#2d5d8a", "#9b3b46", "#9a6b00", "#5b4b8a",
           "#3a7d7b", "#a0522d", "#557a2f", "#8a3b6b", "#3b6ea0"]
print("Готово. Виджеты:", "да" if HAVE_WIDGETS else "нет (графики будут статичными)")

## 2. Данные, на которых будем показывать

- **`digits`** — 1797 рукописных цифр 8×8 (64 признака) со встроенной разметкой. Классика для
  демонстрации DR: видно, как методы раскладывают цифры по «кучкам».
- **Speaker embeddings** — реальные **128-мерные WavLM-эмбеддинги** из датасета
  [`aiacademy-kg/audio-3-speaker-dataset`](https://huggingface.co/datasets/aiacademy-kg/audio-3-speaker-dataset):
  3 диктора (`kore`, `puck`, `andrew`), по 1500 записей. Метки (кто говорит) у нас есть — но
  методам снижения размерности они **не нужны** (это unsupervised), мы используем их только
  чтобы раскрасить картинки и проверить себя.

In [ ]:
# digits
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print("digits:", X_dig.shape, "| классов:", len(np.unique(y_dig)))


def load_speaker_embeddings(candidates=("speaker_emb.npz", "data/speaker_emb.npz")):
    # Load precomputed 128-dim WavLM speaker embeddings (4500 x 128) + labels.
    # Tries a local cache first; otherwise pulls ONLY the embedding/label columns from
    # HuggingFace (skipping the 2.65 GB audio) and caches the result locally. The one-time
    # remote extraction takes ~1-2 minutes; afterwards it is instant and offline.
    import os
    for c in candidates:
        if os.path.exists(c):
            d = np.load(c, allow_pickle=True)
            return d["X"], d["y"], d["text"]
    from huggingface_hub import HfApi
    import pyarrow.parquet as pq
    import fsspec
    repo = "aiacademy-kg/audio-3-speaker-dataset"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    fs = fsspec.filesystem("hf")
    E, Y, T = [], [], []
    for f in files:
        pf = pq.ParquetFile(fs.open(f"datasets/{repo}/{f}"))
        d = pf.read(columns=["wavlm_embedding", "speaker_id", "text"]).to_pandas()
        E.append(np.array(d["wavlm_embedding"].tolist(), dtype=np.float32))
        Y += d["speaker_id"].tolist()
        T += d["text"].tolist()
    X, y, text = np.vstack(E), np.array(Y), np.array(T, dtype=object)
    np.savez_compressed("speaker_emb.npz", X=X, y=y, text=text)
    return X, y, text


X_emb, y_emb, txt_emb = load_speaker_embeddings()
print("embeddings:", X_emb.shape, "| дикторы:", dict(zip(*np.unique(y_emb, return_counts=True))))

In [ ]:
X_dig

## 3. SVD — как найти «главные направления» в данных

> Дальше — самая важная идея всего ноутбука. Разберём её **на пальцах**, без страшных слов.

Представьте облако точек, вытянутое как длинная сигара: вдоль «сигары» точки разбросаны
сильно, поперёк — слабо. Возникает естественный вопрос: **вдоль какого направления данные
разбросаны сильнее всего?** Это направление самое «информативное» — именно по нему точки
больше всего отличаются друг от друга.

**Аналогия с фотографией.** Чтобы сфотографировать вытянутый предмет — например, ложку — есть
удачный ракурс (вдоль длинной стороны, видно максимум) и неудачный (с торца, видно почти
ничего). Поиск «главных направлений» в данных — это и есть поиск самых удачных ракурсов, с
которых, глядя на данные, мы теряем меньше всего.

**Тень (проекция).** «Посмотреть на данные вдоль направления» = уронить каждую точку
перпендикуляром на прямую этого направления, как тень на стену. Насколько широко разбросаны
эти тени — настолько направление информативно: широкие тени — много различий, узкие тени —
точки почти сливаются.

Покрутите направление руками и проследите за разбросом теней (правый график — разброс теней
для каждого угла):

In [ ]:
# A stretched 2D cloud. Rotate a direction and watch how spread-out the shadows are.
rng = np.random.default_rng(1)
cloud = rng.normal(size=(400, 2)) @ np.array([[3.0, 1.4], [0.0, 1.0]])
cloud = cloud - cloud.mean(0)

# Direction (angle) of maximum spread, found by SVD — shown for reference.
_, _, Vt0 = np.linalg.svd(cloud, full_matrices=False)
best_angle = np.degrees(np.arctan2(Vt0[0, 1], Vt0[0, 0])) % 180
angles = np.arange(0, 180, 2)
spreads = np.array([np.var(cloud @ np.array([np.cos(np.deg2rad(a)), np.sin(np.deg2rad(a))]))
                    for a in angles])


def rotate_direction(angle=20):
    d = np.array([np.cos(np.deg2rad(angle)), np.sin(np.deg2rad(angle))])  # unit direction
    proj = cloud @ d                 # shadow coordinate of each point along the direction
    shadows = np.outer(proj, d)      # shadow points lying on the line
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    L = 10
    a1.plot([-L * d[0], L * d[0]], [-L * d[1], L * d[1]], color="#9b3b46", lw=2, zorder=1)
    a1.scatter(cloud[:, 0], cloud[:, 1], s=12, alpha=0.35, color="#2d5d8a", label="точки", zorder=2)
    a1.scatter(shadows[:, 0], shadows[:, 1], s=10, alpha=0.6, color="#9b3b46", label="тени", zorder=3)
    a1.set_xlim(-10, 10); a1.set_ylim(-10, 10); a1.set_aspect("equal")
    a1.legend(loc="upper left")
    a1.set_title(f"Направление {angle}°  ·  разброс теней = {proj.var():.1f}")
    a2.plot(angles, spreads, color="#2f6f4f")
    a2.axvline(angle, color="#9b3b46", lw=2, label="текущий угол")
    a2.axvline(best_angle, color="#2d5d8a", ls="--", label=f"максимум (SVD) ≈ {best_angle:.0f}°")
    a2.set_xlabel("угол направления, °"); a2.set_ylabel("разброс теней (дисперсия)")
    a2.legend(); a2.set_title("Разброс теней зависит от направления")
    plt.show()


if HAVE_WIDGETS:
    interact(rotate_direction, angle=IntSlider(min=0, max=178, step=2, value=20))
else:
    rotate_direction(20)

Где разброс теней максимальный — то самое **первое главное направление**. Следующее ищут уже
среди направлений, **перпендикулярных** первому (чтобы не повторять учтённое), и снова берут
то, где разброс наибольший. И так далее.

Именно это делает **SVD (singular value decomposition, сингулярное разложение)** —
математическая «машина», которая разом находит **все** такие направления, упорядоченные от
самого «широкого» к самому «узкому», и для каждого выдаёт число — **сколько в нём разброса**.
Эти числа называют **сингулярными**, $\sigma_1 \ge \sigma_2 \ge \dots$: большое $\sigma$ —
направление несёт много информации, маленькое — почти плоско (часто это шум).

Записывают SVD как $X = U\Sigma V^{\top}$, и читать это можно по-человечески так:

- $V$ — **список найденных направлений** (взаимно перпендикулярных, длины 1);
- $\Sigma$ — их «ширины» $\sigma_i$ (сколько разброса вдоль каждого);
- $U$ — **координаты точек в новых осях** (где какая точка стоит вдоль этих направлений).

Вот два главных направления нашего облака — длина стрелки показывает разброс ($\sigma$):

In [ ]:
# The two main directions found by SVD. Arrow length = amount of spread along it.
U, S, Vt = np.linalg.svd(cloud, full_matrices=False)
print("сингулярные числа (разброс по направлениям):", np.round(S, 1))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(cloud[:, 0], cloud[:, 1], s=12, alpha=0.35, color="#2d5d8a")
scale = 1.0 / np.sqrt(len(cloud))                      # turn singular value into ~std
for i, c, name in [(0, "#9b3b46", "1-е направление"), (1, "#2f6f4f", "2-е направление")]:
    dx, dy = float(Vt[i, 0] * S[i] * scale), float(Vt[i, 1] * S[i] * scale)
    ax.quiver(0, 0, dx, dy, angles="xy", scale_units="xy", scale=1,
              color=c, width=0.012, label=f"{name} (σ={S[i]:.1f})")
ax.set_aspect("equal"); ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)
ax.legend(loc="upper left"); ax.set_title("SVD: главные направления облака")
plt.show()

### Сжатие: оставляем только самые «широкие» направления

Если оставить лишь несколько направлений с самыми большими $\sigma$, а слабые отбросить,
получится компактное приближение данных — это сразу и **сжатие**, и **шумоподавление**: мы
храним только то, где много различий, и выкидываем «плоские» направления, где в основном шум.
(Математики называют это наилучшим низкоранговым приближением — теорема **Эккарта–Янга**.)

Покажем на картинках цифр: реконструируем их, оставляя только $k$ направлений из 64. При
маленьком $k$ цифры размыты, с ростом $k$ — всё чётче. Подвигайте ползунок:

In [ ]:
# Low-rank reconstruction of digit images via SVD of the centered data matrix
Xc = X_dig - X_dig.mean(0)
Uo, So, Vto = np.linalg.svd(Xc, full_matrices=False)   # So has 64 singular values
energy = np.cumsum(So**2) / np.sum(So**2)

sample_idx = [0, 1, 2, 3, 17, 15, 13, 30, 11, 26]      # one example per digit-ish


def show_reconstruction(k=8):
    Xr = (Uo[:, :k] * So[:k]) @ Vto[:k] + X_dig.mean(0)   # rank-k approximation
    fig, axes = plt.subplots(2, 10, figsize=(13, 3))
    for j, idx in enumerate(sample_idx):
        axes[0, j].imshow(X_dig[idx].reshape(8, 8), cmap="gray_r"); axes[0, j].axis("off")
        axes[1, j].imshow(Xr[idx].reshape(8, 8),   cmap="gray_r"); axes[1, j].axis("off")
    axes[0, 0].set_ylabel("ориг.", rotation=0, labelpad=30);
    fig.suptitle(f"Реконструкция рангом k={k} из 64  ·  сохранено энергии: {energy[k-1]*100:.1f}%",
                 fontsize=13)
    plt.show()


if HAVE_WIDGETS:
    interact(show_reconstruction, k=IntSlider(min=1, max=64, step=1, value=8))
else:
    show_reconstruction(8)

## 4. PCA — метод главных компонент

**PCA (Principal Component Analysis)** — это **SVD, применённый к центрированным данным**
(из каждого признака вычли его среднее). Главные компоненты — те самые правые сингулярные
векторы $V$; они задают новые оси, вдоль которых дисперсия данных максимальна и убывает по
порядку.

**Доля объяснённой дисперсии** каждой компоненты:

$$ \text{explained\_variance\_ratio}_i = \frac{\sigma_i^2}{\sum_j \sigma_j^2}. $$

График этих долей по порядку называют **scree-plot** («каменистая осыпь»): по нему видно,
сколько компонент реально несут сигнал. Часто резкий «локоть» показывает, где сигнал
заканчивается и начинается шум.

In [ ]:
# Scree plot for digits
pca_full = PCA().fit(StandardScaler().fit_transform(X_dig))
ratio = pca_full.explained_variance_ratio_

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.bar(range(1, 21), ratio[:20], color="#2d5d8a")
a1.set_title("Scree-plot (доля дисперсии по компонентам)")
a1.set_xlabel("компонента"); a1.set_ylabel("доля дисперсии")
a2.plot(range(1, len(ratio) + 1), np.cumsum(ratio), marker="o", ms=3, color="#2f6f4f")
a2.axhline(0.9, ls="--", color="#9b3b46"); a2.text(2, 0.91, "90%", color="#9b3b46")
a2.set_title("Накопленная объяснённая дисперсия")
a2.set_xlabel("число компонент"); a2.set_ylabel("накоплено")
plt.show()
print("Компонент для 90% дисперсии:", int(np.argmax(np.cumsum(ratio) >= 0.9) + 1), "из 64")

In [ ]:
# 2D PCA of digits, colored by the (known) digit label
X2 = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_dig))
fig, ax = plt.subplots(figsize=(8, 6))
for d in range(10):
    m = y_dig == d
    ax.scatter(X2[m, 0], X2[m, 1], s=14, alpha=0.6, color=PALETTE[d], label=str(d))
ax.legend(title="цифра", ncol=2, fontsize=9); ax.set_title("digits в 2D (PCA)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.show()

### PCA на реальных эмбеддингах дикторов

Снизим 128 измерений до 2 и до 3 и раскрасим по диктору. Уже **две** главные компоненты
улавливают большую долю дисперсии — этого хватает, чтобы три голоса разъехались в стороны.
3D-график **можно вращать мышью**.

In [ ]:
Xe = StandardScaler().fit_transform(X_emb)
pca_e = PCA(n_components=3).fit(Xe)
Pe = pca_e.transform(Xe)
print("Объяснённая дисперсия 3 компонент:", np.round(pca_e.explained_variance_ratio_[:3], 3),
      " | первые две:", round(pca_e.explained_variance_ratio_[:2].sum(), 3))

fig = px.scatter_3d(x=Pe[:, 0], y=Pe[:, 1], z=Pe[:, 2], color=y_emb,
                    color_discrete_sequence=PALETTE,
                    labels={"x": "PC1", "y": "PC2", "z": "PC3", "color": "диктор"},
                    title="WavLM-эмбеддинги дикторов в 3D (PCA) — вращайте мышью",
                    opacity=0.6)
fig.update_traces(marker_size=3)
fig.update_layout(height=600)
fig.show()

### Где PCA не справляется: нелинейные многообразия

PCA — **линейный** метод: он умеет только поворачивать и проецировать. Если данные лежат на
изогнутой поверхности («рулет»), плоская проекция их **перемешает**. Классический пример —
swiss roll: цвет кодирует положение вдоль скрученного листа, и хочется его «развернуть», но
PCA так не умеет.

In [ ]:
Xsr, color_sr = make_swiss_roll(n_samples=1500, noise=0.05, random_state=0)
Xsr2 = PCA(n_components=2).fit_transform(Xsr)

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax1.scatter(Xsr[:, 0], Xsr[:, 1], Xsr[:, 2], c=color_sr, cmap="Spectral", s=10)
ax1.set_title("Swiss roll (истинное 3D-многообразие)")
ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(Xsr2[:, 0], Xsr2[:, 1], c=color_sr, cmap="Spectral", s=10)
ax2.set_title("PCA в 2D — рулет НЕ развёрнут (цвета перемешаны)")
plt.show()

## 5. t-SNE — нелинейное вложение для визуализации

**t-SNE (t-distributed Stochastic Neighbor Embedding)** решает другую задачу: не сохранить
глобальную дисперсию, а **сохранить локальное соседство**. Идея в три шага:

1. В исходном пространстве близость объектов $i$ и $j$ превращают в **вероятность соседства**
   $p_{ij}$ (гауссово ядро: близкие — вероятные соседи).
2. В целевом 2D-пространстве задают такие же вероятности $q_{ij}$, но через «тяжёлохвостое»
   распределение Стьюдента ($t$) — это спасает от «скучивания» в центре.
3. Точки в 2D двигают так, чтобы $q_{ij}$ как можно лучше повторяли $p_{ij}$ — минимизируют
   расхождение Кульбака–Лейблера $\mathrm{KL}(p \,\|\, q)$ градиентным спуском.

Главный гиперпараметр — **perplexity**: грубо, «сколько соседей учитывать» (обычно 5–50).
Маленькая — внимание к мелким группам, большая — к крупной структуре.

In [ ]:
# t-SNE on digits — local structure pops out as clean islands
Xd_std = StandardScaler().fit_transform(X_dig)
Xt = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xd_std)

fig, ax = plt.subplots(figsize=(8, 6))
for d in range(10):
    m = y_dig == d
    ax.scatter(Xt[m, 0], Xt[m, 1], s=14, alpha=0.7, color=PALETTE[d], label=str(d))
ax.legend(title="цифра", ncol=2, fontsize=9)
ax.set_title("digits в 2D (t-SNE, perplexity=30)")
plt.show()

**Влияние perplexity.** t-SNE медленный, поэтому посчитаем вложения для нескольких значений
заранее (на подвыборке цифр) и переключаемся между ними. Обратите внимание, как меняется
«разрешение» структуры.

In [ ]:
# Precompute t-SNE for a few perplexities on a subset (t-SNE is slow)
sub = np.random.default_rng(0).choice(len(X_dig), 600, replace=False)
Xs_sub = StandardScaler().fit_transform(X_dig[sub]); ys_sub = y_dig[sub]
tsne_by_perp = {p: TSNE(n_components=2, perplexity=p, init="pca", random_state=0).fit_transform(Xs_sub)
                for p in [5, 30, 50]}


def show_tsne(perplexity=30):
    Z = tsne_by_perp[perplexity]
    fig, ax = plt.subplots(figsize=(7, 6))
    for d in range(10):
        m = ys_sub == d
        ax.scatter(Z[m, 0], Z[m, 1], s=16, alpha=0.7, color=PALETTE[d], label=str(d))
    ax.legend(title="цифра", ncol=2, fontsize=8)
    ax.set_title(f"t-SNE, perplexity={perplexity}")
    plt.show()


if HAVE_WIDGETS:
    interact(show_tsne, perplexity=Dropdown(options=[5, 30, 50], value=30))
else:
    show_tsne(30)

In [ ]:
# t-SNE on speaker embeddings (subsample for speed) -> three clean islands
sub_e = np.random.default_rng(0).choice(len(X_emb), 1500, replace=False)
Xe_sub = StandardScaler().fit_transform(X_emb[sub_e]); ye_sub = y_emb[sub_e]
Te = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xe_sub)

fig = px.scatter(x=Te[:, 0], y=Te[:, 1], color=ye_sub, color_discrete_sequence=PALETTE,
                 labels={"x": "t-SNE 1", "y": "t-SNE 2", "color": "диктор"},
                 title="WavLM-эмбеддинги дикторов (t-SNE)", opacity=0.7)
fig.update_traces(marker_size=5)
fig.update_layout(height=550)
fig.show()

### ⚠️ Подводные камни t-SNE

t-SNE — инструмент **визуализации**, а не препроцессинга. Что важно помнить:

- **Расстояния между кластерами не интерпретируемы.** Если две группы далеко на картинке —
  это не значит, что они «сильнее различаются», чем близкие.
- **Размеры кластеров обманчивы.** t-SNE раздувает плотные группы и сжимает разреженные.
- **Стохастичность.** Разные запуски (и `random_state`) дают разные картинки.
- **Не кластеризуйте по координатам t-SNE.** Для кластеризации берите исходные признаки или
  PCA-проекцию; t-SNE-координаты для этого не предназначены.

Практичный конвейер: **сначала PCA** (убрать шум, ускорить), **потом t-SNE** на главных
компонентах — именно так и делают на больших данных.

## 6. PCA против t-SNE: сравнение и когда что брать

| | **PCA / SVD** | **t-SNE** |
|---|---|---|
| тип | линейный | нелинейный |
| что сохраняет | глобальную дисперсию | локальное соседство |
| детерминирован | да | нет (стохастичен) |
| обратим (реконструкция) | да | нет |
| скорость | очень быстро | медленно |
| для чего | сжатие, шумоподавление, препроцессинг | картинка для глаз |

Поставим их рядом на эмбеддингах дикторов:

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
cmap = {s: PALETTE[i] for i, s in enumerate(np.unique(ye_sub))}
cols = [cmap[s] for s in ye_sub]
Pe2 = PCA(n_components=2).fit_transform(Xe_sub)
a1.scatter(Pe2[:, 0], Pe2[:, 1], c=cols, s=12, alpha=0.6); a1.set_title("PCA (2D)")
a2.scatter(Te[:, 0], Te[:, 1], c=cols, s=12, alpha=0.6); a2.set_title("t-SNE (2D)")
for s, c in cmap.items():
    a1.scatter([], [], color=c, label=s)
a1.legend(title="диктор")
plt.show()

## 7. Выводы

- **SVD** $X = U\Sigma V^\top$ — общий движок: даёт направления и их важность (сингулярные
  числа). Низкоранговое приближение (Эккарт–Янг) — это сжатие и шумоподавление.
- **PCA** = SVD на центрированных данных. Главные компоненты максимизируют дисперсию;
  scree-plot показывает, сколько их реально нужно. Быстрый, обратимый, отличный
  препроцессинг — но **линейный** (рулет не развернёт).
- **t-SNE** сохраняет соседство и даёт красивые «острова» для глаз, но капризен: расстояния и
  размеры не читать, по его координатам не кластеризовать.
- На реальных **128-мерных эмбеддингах** обе техники показали три чётко разделённых диктора —
  значит, в данных есть сильная структура.

**Мост к следующему ноутбуку.** Мы научились эту структуру *видеть*. Дальше научимся её
*находить алгоритмически*, не подглядывая в метки, — это **кластеризация**. Начнём с общей
постановки задачи, метрик качества и метода **K-Means**.